# Network Security Traffic Analyzer — Public Demo

**Quick start:** Open in Colab, sign in if prompted, review the code, then select **Runtime → Run all**. Approve the external notebook warning only if you trust the code.

No ZIP upload or local installation is required in Colab. Internet access is needed. The final section shows the analysis and regression results, with an optional JSON download.

All inputs are synthetic. Educational prototype, not a production IDS.

### 1. Environment setup
Colab downloads and verifies the pinned public baseline. Local Jupyter uses the project directory.

In [ ]:
from pathlib import Path
import sys, subprocess, tempfile
from importlib.util import find_spec

BASELINE = "28829fc98de74a22a70ac8371256f460d5788bf6"
REPO = "https://github.com/GitMasterMindX/Network_Security_Traffic_Analyzer.git"
try:
    IN_COLAB = find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

report = output = test_result = None
if IN_COLAB:
    root = Path(tempfile.mkdtemp(prefix="network_security_")) / "repo"
    subprocess.run(["git", "clone", "--quiet", "--no-checkout", REPO, str(root)], check=True)
    subprocess.run(["git", "-C", str(root), "checkout", "--quiet", "--detach", BASELINE], check=True)
    actual = subprocess.check_output(["git", "-C", str(root), "rev-parse", "HEAD"], text=True).strip()
    if actual != BASELINE:
        raise RuntimeError("Baseline verification failed")
else:
    root = Path.cwd()
    if not (root / "analyzer_v04.py").is_file():
        root = root / "Network_Security_Traffic_Analyzer_Public_Release"
for required in ("analyzer_v04.py", "synthetic_data", "tests"):
    if not (root / required).exists():
        raise FileNotFoundError(f"Missing {required}; launch from the extracted project folder")
sys.path.insert(0, str(root))
print("Project:", root)


### 2. Parse the synthetic network logs

In [ ]:
from analyzer_v04 import extract_tcpdump, extract_wireshark, summarize, save
sources={'dns_icmp':('dns_icmp.txt',extract_tcpdump),'http':('http.txt',extract_tcpdump),'syn':('syn.txt',extract_wireshark)}
evidence={}
for case,(filename,parser) in sources.items():
    events,rejected=parser((root/'synthetic_data'/filename).read_text(encoding='utf-8'),filename)
    evidence[case]={'events':events,'rejected':rejected}
    print(case,'parsed:',len(events),'rejected:',len(rejected))


### 3. Detect patterns and generate the JSON report

In [ ]:
report=summarize(evidence)
report['evidence_origin']='synthetic_demo'
report['limitations']=['Synthetic demonstration, not evidence of a real incident.']
for case,item in report['reports'].items():
    print(case,item['counts'],[(f['id'],f['classification']) for f in item['findings']])
output=root/'reports'/'synthetic_demo_report.json'
output.parent.mkdir(exist_ok=True)
save(report,output)
print('Saved:',output)


### 4. Automated regression tests
Install pytest if missing, then run the public regression suite.

In [ ]:
import importlib.util
import re
import time

test_result = None
if importlib.util.find_spec("pytest") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "pytest==9.2.0"], check=True)
start = time.perf_counter()
test_result = subprocess.run([sys.executable, "-m", "pytest", "-q", "-o", "addopts="], cwd=root, capture_output=True, text=True)
print(test_result.stdout)
print(test_result.stderr)
print(f"Elapsed: {time.perf_counter() - start:.2f} s | Exit code: {test_result.returncode}")


### 5. Final results
The status below is calculated from the generated report and actual test output.

In [ ]:
from IPython.display import display, Markdown, FileLink
import json

if report is None or output is None or test_result is None:
    raise RuntimeError("Run all preceding cells first.")
items = report["reports"]
parsed = sum(x["parsed"] for x in items.values())
rejected = sum(x["rejected"] for x in items.values())
findings = [f for x in items.values() for f in x["findings"]]
observed = sum(f["classification"] == "observed" for f in findings)
suspicious = sum(f["classification"] == "suspicious" for f in findings)
match = re.search(r"\b(\d+) passed\b", test_result.stdout)
passed = int(match.group(1)) if match else 0
report_ok = output.is_file() and json.loads(output.read_text(encoding="utf-8")) == report
ok = (test_result.returncode == 0 and passed == 6 and report_ok
      and set(items) == {"dns_icmp", "http", "syn"}
      and parsed == 12 and rejected == 0 and len(findings) == 6
      and observed == 5 and suspicious == 1
      and report.get("evidence_origin") == "synthetic_demo")
display(Markdown(f"""## {'PASS — Demonstration completed' if ok else 'FAIL — Review the results'}

| Metric | Result |
|---|---:|
| Scenarios | {len(items)}/3 |
| Parsed records | {parsed}/12 |
| Rejected records | {rejected} |
| Observed findings | {observed} |
| Suspicious findings | {suspicious} |
| Regression tests passed | {passed}/6 |
| Test exit code | {test_result.returncode} |
| JSON report verified | {report_ok} |

A suspicious pattern is not proof of an attack. Synthetic data only.
"""))
if not ok:
    raise RuntimeError("Demonstration acceptance checks failed.")
if not IN_COLAB:
    display(FileLink(str(output.relative_to(Path.cwd())), result_html_prefix="Download JSON report: "))


### 6. Optional JSON download (Colab)
Run the following cell to define the download helper. Then run `download_report()` in a new code cell only if you want the file. The demo does not trigger automatic downloads.

In [ ]:
def download_report():
    if not globals().get("ok", False):
        raise RuntimeError("Complete the acceptance checks first.")
    if IN_COLAB:
        from google.colab import files
        files.download(str(output))
    else:
        display(FileLink(str(output.relative_to(Path.cwd()))))

if IN_COLAB and globals().get("ok", False):
    try:
        import ipywidgets as widgets
        button = widgets.Button(description="Download JSON report", layout=widgets.Layout(width="220px"))
        button.on_click(lambda _: download_report())
        display(button)
    except Exception:
        print("Optional button unavailable; run download_report() in a new cell.")
